# Model Monitor

Model Monitor baseline works by profiling a dataset to establish what "normal" data looks like, then in production it alerts if new data drifts from that baseline.
  
We have to run the baseline on the training data and monitor the production simulation data against it.

In [3]:
import boto3
import sagemaker
import awswrangler as wr
from sagemaker.core.helper.session_helper import Session, get_execution_role

sess = Session()
bucket = sess.default_bucket()

# setup the IAM, so it gives featurestore access to the s3 bucket
role = get_execution_role()
region = sess.boto_region_name

In [7]:
%store -r s3_aerodelay

 # Model Monitor Baseline — AeroDelay AI
  
Sets up a SageMaker Model Monitor baseline from the training data.
The baseline captures statistics and constraints that production data will be monitored against.

## What this produces
- `statistics.json` — per-feature stats (mean, std, min, max, distribution)
- `constraints.json` — rules for what "normal" data looks like

## Why
When teammates run batch predictions on production simulation data,
Model Monitor compares it against this baseline to detect data drift.

  ## Prepare Baseline Dataset
  
  - Load training data and drop the target column `arrdel15`.
 -  Model Monitor baselines features only, not the label.

In [8]:

# Load training data as the baseline
train_df = wr.s3.read_parquet(f"{s3_aerodelay}/training/data.parquet")

# Drop target — monitor only features, not the label
baseline_df = train_df.drop(columns=["arrdel15"])

print(f"Baseline shape: {baseline_df.shape}")
print(f"Columns: {list(baseline_df.columns)}")


Baseline shape: (650420, 19)
Columns: ['year', 'month', 'dayofmonth', 'dayofweek', 'reporting_airline', 'origin', 'dest', 'originstate', 'deststate', 'crselapsedtime', 'distance', 'dep_hour', 'is_weekend', 'route', 'carrier_delay_rate', 'origin_delay_rate', 'dest_delay_rate', 'route_delay_rate', 'hour_delay_rate']


## Save Baseline to S3

In [9]:
import time

# Save baseline to S3
baseline_s3_path = f"{s3_aerodelay}/model-monitor/baseline-input/baseline.csv"
wr.s3.to_csv(
  df=baseline_df,
  path=baseline_s3_path,
  index=False,
)
print(f"Baseline saved to {baseline_s3_path}")

Baseline saved to s3://sagemaker-us-east-1-103012382341/airline-delay/model-monitor/baseline-input/baseline.csv


 ## Run Baseline Job
  
SageMaker Model Monitor analyzer container profiles the baseline dataset and produces `statistics.json` and `constraints.json`.

It runs a pre-built AWS container that reads our CSV and automatically calculates:
  
  For each numeric feature (like distance, dep_hour):
  - Mean, standard deviation, min, max
  - Quantiles (25%, 50%, 75%) 
  - Missing value count

  For each categorical feature (like origin, route):
  - Number of unique values
  - Most common values
  
  Then it writes two output files:

  statistics.json :the actual numbers. Example:
  {
    "name": "dep_hour",
    "mean": 13.2,
    "std_dev": 4.8,
    "min": 0,
    "max": 23
  } 
  
  constraints.json — the rules derived from those numbers. Example:
  {
    "name": "dep_hour",
    "completeness": 1.0,
    "num_constraints": {
      "is_non_negative": true
    } 
  } 
  
Later when production data comes in, Model Monitor runs the same profiling on it and compares against these files. If dep_hour mean is suddenly 18 instead of 13, or if origin has airport codes that never appeared in training — it flags those as violations.

In [10]:
from sagemaker.core.processing import Processor
sm_client = boto3.client("sagemaker")

baseline_job_name = f"aerodelay-baseline-{int(time.time())}"
baseline_output_s3 = f"{s3_aerodelay}/model-monitor/baseline-output/"

response = sm_client.create_processing_job(
  ProcessingJobName=baseline_job_name,
  ProcessingResources={
      "ClusterConfig": {
          "InstanceCount": 1,
          "InstanceType": "ml.m5.xlarge",
          "VolumeSizeInGB": 20,
      }
  },
  AppSpecification={
      "ImageUri": f"156813124566.dkr.ecr.{region}.amazonaws.com/sagemaker-model-monitor-analyzer",
  },
  ProcessingInputs=[
      {
          "InputName": "baseline_input",
          "S3Input": {
              "S3Uri": baseline_s3_path,
              "LocalPath": "/opt/ml/processing/input/baseline",
              "S3DataType": "S3Prefix",
              "S3InputMode": "File",
          },
      }
  ],
  ProcessingOutputConfig={
      "Outputs": [
          {
              "OutputName": "baseline_output",
              "S3Output": {
                  "S3Uri": baseline_output_s3,
                  "LocalPath": "/opt/ml/processing/output",
                  "S3UploadMode": "EndOfJob",
              },
          }
      ]
  },
  Environment={
      "dataset_format": '{"csv": {"header": true}}',
      "dataset_source": "/opt/ml/processing/input/baseline",
      "output_path": "/opt/ml/processing/output",
      "publish_cloudwatch_metrics": "Disabled",
  },
  RoleArn=role,
)

print(f"Baseline job started: {baseline_job_name}")

# Wait for completion
while True:
  status = sm_client.describe_processing_job(
      ProcessingJobName=baseline_job_name
  )["ProcessingJobStatus"]
  print(f"  Status: {status}")
  if status in ("Completed", "Failed", "Stopped"):
      break
  time.sleep(30)

print(f"Final status: {status}")

Baseline job started: aerodelay-baseline-1780275684
  Status: InProgress


  Status: InProgress


  Status: InProgress


  Status: InProgress


  Status: InProgress


...

  Status: InProgress


  Status: InProgress


  Status: InProgress


  Status: InProgress


  Status: Completed
Final status: Completed


In [12]:
files = wr.s3.list_objects(baseline_output_s3)
for f in files:
  print(f)

s3://sagemaker-us-east-1-103012382341/airline-delay/model-monitor/baseline-output/constraints.json
s3://sagemaker-us-east-1-103012382341/airline-delay/model-monitor/baseline-output/statistics.json


In [15]:
import json
import boto3

s3 = boto3.client("s3")

# Read statistics.json
stats = s3.get_object(
  Bucket=bucket,
  Key="airline-delay/model-monitor/baseline-output/statistics.json"
)
stats_json = json.loads(stats["Body"].read())

# Read constraints.json
constraints = s3.get_object(
  Bucket=bucket,
  Key="airline-delay/model-monitor/baseline-output/constraints.json"
)
constraints_json = json.loads(constraints["Body"].read())

# Show statistics for first 3 features
print("=== STATISTICS (first 3 features) ===")
for feature in stats_json["features"][:1]:
  print(json.dumps(feature, indent=2))

print("\n=== CONSTRAINTS (first 3 features) ===")
for feature in constraints_json["features"][:3]:
  print(json.dumps(feature, indent=2))

=== STATISTICS (first 3 features) ===
{
  "name": "year",
  "inferred_type": "Integral",
  "numerical_statistics": {
    "common": {
      "num_present": 650420,
      "num_missing": 0
    },
    "mean": 2024.0,
    "sum": 1316450080.0,
    "std_dev": 0.0,
    "min": 2024.0,
    "max": 2024.0,
    "approximate_num_distinct_values": 1,
    "completeness": 1.0,
    "distribution": {
      "kll": {
        "buckets": [
          {
            "lower_bound": 2024.0,
            "upper_bound": 2024.0,
            "count": 0.0
          },
          {
            "lower_bound": 2024.0,
            "upper_bound": 2024.0,
            "count": 0.0
          },
          {
            "lower_bound": 2024.0,
            "upper_bound": 2024.0,
            "count": 0.0
          },
          {
            "lower_bound": 2024.0,
            "upper_bound": 2024.0,
            "count": 0.0
          },
          {
            "lower_bound": 2024.0,
            "upper_bound": 2024.0,
            "count